# FinSafe 06 — unattended first-submission experiment
Save & Run All on Kaggle T4 with Internet and enabled HF_TOKEN plus HF_WRITE (or HF_WRITE_TOKEN). Pinned source ef272c570f62c07d680fd295df4812167e018c1d. Runs real GPU steering preflight, 100-step finance SFT, matched aligned/SFT/steering validation, gated preliminary AI review, AuditKIT and explicit public HF publication. Full-state checkpoints and phase archives go to private HF recovery. Final organizer-form submission remains a separate action. Server-side execution continues during laptop sleep; Codex/browser work may pause. Read docs/UNATTENDED_RUN.md.


In [ ]:
import os,sys,json,subprocess,time,datetime,hashlib,zipfile
from pathlib import Path
boot=time.monotonic()
print(datetime.datetime.now(datetime.timezone.utc).isoformat(),"START unattended pipeline bootstrap",flush=True)
try:
    project=Path('/kaggle/working/finsafe')
    revision='ef272c570f62c07d680fd295df4812167e018c1d'
    if not project.exists():subprocess.run(['git','clone','https://github.com/dixitkeshav/finsafe.git',str(project)],check=True)
    subprocess.run(['git','-C',str(project),'fetch','origin',revision],check=True)
    subprocess.run(['git','-C',str(project),'checkout','--detach',revision],check=True)
    sys.path.insert(0,str(project/'src'))
    from finsafe.progress import Progress,atomic_json
    progress=Progress(project/'runs/submission-bootstrap/logs','bootstrap')
    subprocess.run([sys.executable,'-m','pip','install','-q','uv','huggingface_hub==1.33.0','kagglehub'],check=True)
    from kaggle_secrets import UserSecretsClient
    secrets=UserSecretsClient()
    read_token=secrets.get_secret('HF_TOKEN')
    try:write_token=secrets.get_secret('HF_WRITE_TOKEN')
    except Exception:write_token=secrets.get_secret('HF_WRITE')
    env=os.environ.copy()
    env.update(HF_TOKEN=read_token,HF_WRITE_TOKEN=write_token,HF_HOME='/kaggle/temp/finsafe-hf',CUDA_VISIBLE_DEVICES='0',
      TOKENIZERS_PARALLELISM='false',TRL_ONLY_MODE='1',DISABLE_UNSLOTH_FOR_TRL='1',PYTHONUNBUFFERED='1',
      UV_CACHE_DIR='/kaggle/temp/finsafe-uv-cache',UV_PYTHON_INSTALL_DIR='/kaggle/temp/finsafe-python',UV_HTTP_TIMEOUT='180')
    from huggingface_hub import HfApi,hf_hub_download
    api=HfApi(token=write_token)
    with progress.stage('01-auth-and-restore'):
        progress.log('HF identity: '+api.whoami()['name'])
        repo='Finsafe-team/finsafe-recovery'
        api.create_repo(repo,repo_type='dataset',private=True,exist_ok=True)
        if not api.repo_info(repo,repo_type='dataset').private:raise RuntimeError('Recovery repo must be private')
        restore='latest/submission-recovery.zip' if api.file_exists(repo,'latest/submission-recovery.zip',repo_type='dataset') else 'bootstrap/version5-recovery.zip'
        archive=hf_hub_download(repo,restore,repo_type='dataset',token=write_token)
        if restore.startswith('bootstrap') and hashlib.sha256(Path(archive).read_bytes()).hexdigest()!='db8e2cbf4e40ad92ebf48b99ad7e3492ac5d60c03757d415db1d10d04c360ce0':raise RuntimeError('Bootstrap recovery SHA mismatch')
        with zipfile.ZipFile(archive) as z:
            if z.testzip():raise RuntimeError('Archive CRC failure')
            for member in z.infolist():
                path=Path(member.filename)
                if path.is_absolute() or '..' in path.parts or not path.parts or path.parts[0] not in {'data','runs','checkpoints'} or ((member.external_attr>>16)&0o170000)==0o120000:raise RuntimeError('Unsafe archive member')
            z.extractall(project)
        progress.log('Verified portable recovery archive restored')
    with progress.stage('02-python-and-pinned-gpu-environment'):
        gpu_py=str(project/'.venv-gpu/bin/python')
        uv=[sys.executable,'-m','uv']
        if not Path(gpu_py).exists():progress.run(uv+['venv','--python','3.12',str(project/'.venv-gpu')],cwd=project,env=env)
        progress.run(uv+['pip','install','--python',gpu_py,'torch==2.11.0','torchvision==0.26.0','--index-url','https://download.pytorch.org/whl/cu130'],cwd=project,env=env)
        progress.run(uv+['pip','install','--python',gpu_py,'-r','requirements-gpu.txt','-c','constraints-kaggle-t4.txt','transformers==5.23.0','peft==0.21.3','accelerate==1.15.0','datasets==5.2.0','bitsandbytes==0.50.2','huggingface-hub==1.33.0'],cwd=project,env=env)
        progress.run(uv+['pip','check','--python',gpu_py],cwd=project,env=env)
    with progress.stage('03-execute-submission-pipeline'):
        progress.run([gpu_py,'-u','scripts/cloud_pipeline.py'],cwd=project,env=env)
    with progress.stage('04-retain-portable-versioned-output'):
        import shutil
        # Only reproducible environments/caches are removed. Unique data, adapters,
        # trainer checkpoints, logs, reports and archives remain retained.
        for generated in [project/'.venv-gpu',project/'.cache']:
            if generated.exists():shutil.rmtree(generated)
    progress.log('DONE unattended run. Public artifact links are in runs/submission-v1/public-links.json; organizer form still requires actual submission.')
except BaseException as error:
    print(datetime.datetime.now(datetime.timezone.utc).isoformat(),'FAILED unattended run',type(error).__name__,'elapsed',round(time.monotonic()-boot,1),flush=True)
    raise
else:
    print(datetime.datetime.now(datetime.timezone.utc).isoformat(),'DONE notebook elapsed',round(time.monotonic()-boot,1),'seconds',flush=True)
